# ReAct Agents with LangGraph, Groq and Hugging Face

**ReAct = Reason + Act.** Instead of answering in one shot, the LLM works in a loop:

1. **Reason**: read the conversation and decide what to do next
2. **Act**: call a tool (search, calculator, retriever...)
3. **Observe**: the tool's output is added back to the conversation as a `ToolMessage`
4. Repeat until the model has enough information, then answer directly

```
START ──► LLM (reason) ──tool call?──yes──► Tools (act) ──► observation back to LLM
              │
              no
              ▼
             END
```

In this notebook:
- **Groq** (`ChatGroq`) is the reasoning LLM that decides which tool to call
- **Hugging Face** `all-MiniLM-L6-v2` embeddings power a FAISS retriever tool (a small RAG knowledge base)
- **Hugging Face** `ChatHuggingFace` is shown as a drop-in alternative LLM
- **LangGraph** wires the loop together, and `MemorySaver` gives the agent memory across turns

### 1. Install extra packages
`arxiv` is needed by the Arxiv tool and `langchain-tavily` by the web search tool. Everything else is in `requirements.txt`.

In [ ]:
# arxiv 3+ breaks the LangChain Arxiv wrapper, so pin an older version
%pip install -q "arxiv<2.2" langchain-tavily

### 2. Load API keys
`GROQ_API_KEY` is required. `TAVILY_API_KEY` (web search) and `HF_TOKEN` (Hugging Face hosted LLM, section 11) are optional.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Map the key names used in .env to the names the libraries expect
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") or os.getenv("groq_api", "")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY") or os.getenv("tavily_api", "")

# Optional: LangSmith tracing, so every Reason/Act step shows up in the LangSmith UI
if os.getenv("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = "ReAct-agent"

print("Groq key loaded:  ", bool(os.environ["GROQ_API_KEY"]))
print("Tavily key loaded:", bool(os.environ["TAVILY_API_KEY"]))
print("HF token loaded:  ", bool(os.getenv("HF_TOKEN")))

### 3. Built-in tools: Arxiv and Wikipedia
A tool is a function plus a **name** and **description**. The LLM never runs the tool itself; it only reads the name/description and asks for it to be called with some arguments.

In [ ]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
import wikipedia

# Wikipedia rate-limits (HTTP 429) the library's shared default user agent, so identify ourselves
wikipedia.set_user_agent("RAG-ReAct-Notebook/1.0 (learning project)")

arxiv = ArxivQueryRun(api_wrapper=ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=500))
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500))

print(arxiv.name, "->", arxiv.description[:80], "...")
print(wiki.name, "->", wiki.description[:80], "...")

In [ ]:
arxiv.invoke("Attention is all you need")

In [ ]:
wiki.invoke("What is machine learning")

### 4. Custom tools
The `@tool` decorator turns a plain Python function into a tool. The **type hints** become the argument schema and the **docstring** becomes the description the LLM reads, so write them carefully.

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Add two numbers a and b."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers a and b."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

print(multiply.name, multiply.args)

### 5. RAG retriever tool with Hugging Face embeddings
This is where RAG meets agents: the knowledge base becomes just another tool. The agent decides **when** retrieval is needed instead of retrieving on every question.

`all-MiniLM-L6-v2` runs locally on CPU, so no API key is needed.

In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

docs = [
    Document(page_content="Acme Corp's refund policy: customers can return any product within 30 days of purchase for a full refund."),
    Document(page_content="Acme Corp's support team is available Monday to Friday, 9 AM to 6 PM IST, at support@acme.example."),
    Document(page_content="Acme Corp was founded in 2015 in Bengaluru and builds smart home devices."),
    Document(page_content="The Acme SmartPlug costs 1,499 INR and the Acme SmartBulb costs 799 INR."),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

@tool
def acme_knowledge_base(query: str) -> str:
    """Search Acme Corp's internal documents. Use this for any question about Acme Corp:
    its products, prices, refund policy, support hours or company history."""
    results = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in results)

acme_knowledge_base.invoke("How long do I have to return a product?")

### 6. Web search tool (Tavily, optional)
Tavily gives the agent access to live web results. It is skipped automatically if no `TAVILY_API_KEY` is set.

In [ ]:
tools = [arxiv, wiki, acme_knowledge_base, add, multiply, divide]

if os.environ["TAVILY_API_KEY"]:
    from langchain_tavily import TavilySearch
    tavily = TavilySearch(max_results=3)
    tools.append(tavily)

print([t.name for t in tools])

### 7. Groq LLM with tools bound
`bind_tools` sends the tool schemas to the model with every request. When the model wants a tool it returns an `AIMessage` with empty `content` and a filled `tool_calls` list. **This is the "Act" step: the model only *requests* the call.**

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langchain_core.messages import HumanMessage

response = llm_with_tools.invoke([HumanMessage(content="What is 12 multiplied by 7?")])
response.tool_calls

In [ ]:
# A question that needs no tool: the model answers directly, tool_calls is empty
response = llm_with_tools.invoke([HumanMessage(content="Say hello in French")])
print("tool_calls:", response.tool_calls)
print("content:   ", response.content)

### 8. State schema
The graph's state is just the list of messages. `add_messages` is a **reducer**: when a node returns new messages they are appended to the list instead of replacing it. That growing list is the agent's scratchpad of thoughts, actions and observations.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

### 9. Build the ReAct graph
- `tool_calling_llm` node: **Reason**, calls the LLM on the whole message history
- `tools` node (`ToolNode`): **Act**, executes every requested tool call and returns `ToolMessage`s
- `tools_condition`: routes to `tools` if the last message has tool calls, otherwise to `END`
- `tools -> tool_calling_llm` edge: **Observe**, sends tool results back to the LLM. This edge is what makes it a loop (plain tool calling would stop after one tool run)

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### 10. Run the agent
Watch the loop in the printed messages: Human → AI (tool calls) → Tool results → AI (more tool calls, using the earlier results) → AI final answer.

In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="Add 5 plus 5, then multiply the result by 10")]})
for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="What is Acme's refund policy, and how much do a SmartPlug and a SmartBulb cost together?")]})
for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="What is machine learning?")]})
for m in result["messages"]:
    m.pretty_print()

### 11. The memory problem
Each `graph.invoke` starts from an empty state, so the agent forgets the previous turn.

In [ ]:
graph.invoke({"messages": [HumanMessage(content="What is 5 plus 8?")]})["messages"][-1].pretty_print()
graph.invoke({"messages": [HumanMessage(content="Divide that by 5")]})["messages"][-1].pretty_print()

### 12. Agent memory with MemorySaver
A **checkpointer** saves the graph state after every step. Calls with the same `thread_id` continue the same conversation; a different `thread_id` is a fresh conversation. `MemorySaver` keeps it in RAM (use `SqliteSaver` / `PostgresSaver` to persist to disk).

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph_memory = builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "1"}}

In [ ]:
result = graph_memory.invoke({"messages": [HumanMessage(content="Add 12 and 13.")]}, config=config)
result["messages"][-1].pretty_print()

In [ ]:
result = graph_memory.invoke({"messages": [HumanMessage(content="Add that number to 25")]}, config=config)
result["messages"][-1].pretty_print()

In [ ]:
result = graph_memory.invoke({"messages": [HumanMessage(content="Then multiply that number by 2")]}, config=config)
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# A new thread_id has no history
result = graph_memory.invoke({"messages": [HumanMessage(content="What number did we end up with?")]},
                             config={"configurable": {"thread_id": "2"}})
result["messages"][-1].pretty_print()

### 13. Swap in a Hugging Face LLM (optional)
The graph does not care which chat model is inside the node. `ChatHuggingFace` calls a model hosted on Hugging Face Inference Providers and supports `bind_tools`. Needs a free `HF_TOKEN` from huggingface.co/settings/tokens in your `.env`.

In [ ]:
if os.getenv("HF_TOKEN"):
    from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

    hf_llm = ChatHuggingFace(llm=HuggingFaceEndpoint(
        repo_id="Qwen/Qwen2.5-7B-Instruct",
        task="text-generation",
        max_new_tokens=512,
        temperature=0.1,
    ))
    hf_llm_with_tools = hf_llm.bind_tools(tools)

    def hf_tool_calling_llm(state: State):
        return {"messages": [hf_llm_with_tools.invoke(state["messages"])]}

    hf_builder = StateGraph(State)
    hf_builder.add_node("tool_calling_llm", hf_tool_calling_llm)
    hf_builder.add_node("tools", ToolNode(tools))
    hf_builder.add_edge(START, "tool_calling_llm")
    hf_builder.add_conditional_edges("tool_calling_llm", tools_condition)
    hf_builder.add_edge("tools", "tool_calling_llm")
    hf_graph = hf_builder.compile()

    result = hf_graph.invoke({"messages": [HumanMessage(content="What are Acme's support hours?")]})
    for m in result["messages"]:
        m.pretty_print()
else:
    print("Set HF_TOKEN in .env to run the Hugging Face version")

### 14. Shortcut: the prebuilt agent
Everything in sections 8, 9 and 12 is common enough that LangChain ships it as `create_agent`. It builds the same Reason → Act → Observe loop for you. Building it by hand first is how you learn what it does and how to customise it.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools when they help; never do arithmetic in your head.",
    checkpointer=MemorySaver(),
)

cfg = {"configurable": {"thread_id": "prebuilt-1"}}
agent.invoke({"messages": [HumanMessage(content="Who founded Acme and in which year? How many years ago was that from 2026?")]}, cfg)["messages"][-1].pretty_print()